In [ ]:
# Colab setup — fetches the wind data from GitHub. Skip this cell if t07_wind.npz is already in this folder.
!wget -q https://raw.githubusercontent.com/lalcayaga/micromet-lecture7/main/t07_wind.npz -O t07_wind.npz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
np.seterr(divide="ignore", invalid="ignore")   # the ω = 0 bin is zero once the mean is removed

d = np.load("t07_wind.npz")
u = d["u"].astype(float)                 # fine box: shape (Nx, Ny, Nrows), x along the wind, masts 3 m apart across it
report_hub = d["report_hub"].astype(float)   # the report's box at the hub, as its simulation sees it (same time step)
dx, dy, U = float(d["dx"]), float(d["dy"]), float(d["U"])
Nx, Ny, Nrows = u.shape
print(f"fine box: {Nx} points along x (dx = {dx} m), {Nrows} rows of {Ny} masts (dy = {dy} m), U = {U} m/s")
print(f"report's box: {int(d['report_N'])} points along x, dx = {float(d['report_dx'])} m (header)")

# Given (Lecture 6, Round 2): course convention, two-sided, angular frequency.
def fourier(x, dt):
    """x: (N,) or (N, ...) -> omega (N//2+1,), X (N//2+1, ...) for omega >= 0, X = (dt/2pi) FFT."""
    X = np.fft.rfft(x - x.mean(axis=0), axis=0) * dt / (2 * np.pi)
    return 2 * np.pi * np.fft.rfftfreq(x.shape[0], dt), X

def coherence(X, j, T):
    """Given (beyond the notes: estimation): coherence of masts j apart, averaged over all pairs in all rows."""
    S = (2 * np.pi / T) * (np.abs(X) ** 2).mean(axis=(1, 2))
    chi = (2 * np.pi / T) * (X[:, :-j, :] * np.conj(X[:, j:, :])).mean(axis=(1, 2))
    return np.abs(chi) ** 2 / S ** 2

In [ ]:
# Optional: smoother plots with YOUR log_smooth(omega, S, n) from A3 Part A3.1.
# Paste it here and set smooth = log_smooth. Without it (smooth = None) the plots show the raw, noisier spectra.
smooth = None

def show(x, y, n=15):
    # plot helper: x > 0 only; smoothed with your A3 function if you set one, raw otherwise
    m = x > 0
    return smooth(x[m], y[m], n) if smooth else (x[m], y[m])

# Round 1 — The lab bench: rebuilding the wind T07 felt
### 46100 Micrometeorology · Lecture 7 · Patient T07

**Why this matters.** A turbine does not feel "turbulence": it feels the atmosphere's spectrum, reshaped link by link into
thrust, tower motion and what the sensors measure. Today rebuilds that chain; this round is about its first link, the wind itself.

![The load chain](https://raw.githubusercontent.com/lalcayaga/micromet-lecture7/main/load_chain_diagram.png)

**Work in your pair.** The report says its wind is right: the box carries the site's σ_u. Here you have a fine turbulence box
at T07's site (8 rows of 64 virtual met masts, 3 m apart) **and** the report's own box at the hub, as its simulation fed it to
the turbine. Your lab results go into the game.

**Time budget: ~25 min analysis, ~5 min for your group's two numbers (30' total).**

| Task | Time | What you'll do |
|---|---|---|
| 1 | 3' | Turn the box into time series (Taylor) |
| 2 | 7' | Time → space: bet, then check what happens at half the wind speed |
| 3 | 8' | The report's box against the fine box (commit first) → **lab result 1** |
| 4 | 7' | Coherence across the rotor: is Davenport right? → **lab result 2** |

**File → Save a copy in Drive** before writing anything, then share the copy with your partner (Editor).

## Task 1 — Turn the box into time series (3')

The box is frozen: a mast sees it pass at speed $U$ (Taylor, eq. 7.1.3). Compute the time step `dt` between samples and the
record length `T` in seconds.

In [ ]:
# your code here
dt = None
T = None

print(f"dt = {dt} s,  T = {T} s")
plt.figure(figsize=(9, 2.5)); plt.plot(np.arange(Nx) * dt, U + u[:, 0, 0], lw=0.6)
plt.xlabel("t [s]"); plt.ylabel("U + u [m/s]"); plt.title("one mast"); plt.show()

## Task 2 — Time → space (7')

**Bet first** (before running): if the same frozen field passed the masts at **half** the wind speed, would the peak of
$\omega S(\omega)$ move? Would the peak of $k_1 F(k_1)$ move?

*Our bet:* …

Compute $\omega S(\omega)$ averaged over all masts and convert it to space, $k_1 = \omega/U$ and $k_1 F(k_1) = \omega S(\omega)$
(eq. 7.1.9 in the course convention), for $U$ and for $U/2$ (only `dt` changes).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for Ux, lab in [(U, "U"), (U / 2, "U/2")]:
    dt_x = dx / Ux; T_x = Nx * dt_x
    om, X = fourier(u, dt_x)
    # spectrum of every mast, S = (2π/T)|X|² (course convention), averaged over all 64 × 8 masts:
    # averaging over many masts plays the role of averaging over time segments, so the result is smooth
    S_mean = ((2 * np.pi / T_x) * np.abs(X) ** 2).mean(axis=(1, 2))
    # your code here: wavenumber k1 and the premultiplied spatial spectrum k1*F(k1)
    k1 = None
    k1F = None
    ax[0].semilogx(*show(om, om * S_mean), "-", lw=1, label=lab)
    ax[1].semilogx(*show(k1, k1F), "-", lw=1, label=lab)
ax[0].set_xlabel("ω [rad/s]"); ax[0].set_ylabel("ω S(ω)"); ax[1].set_xlabel("k₁ [rad/m]"); ax[1].set_ylabel("k₁ F(k₁)")
for a_ in ax: a_.legend(); a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Task 3 — The report's box against the fine box (8')

Both boxes carry the same σ_u. **Commit first:** will their spectra at the hub match at all frequencies? If not, where?

*Our commit:* …

Compute the spectrum of `report_hub` and the fine box's spectrum averaged over all masts, plot both, and find the angular
frequency above which the report's box has less than **a tenth** of the fine box's energy. Read it from the ratio plot:
the raw ratio is noisy, so look for where it drops *and stays* below 0.1.

**Lab result 1:** ω_silent ≈ … rad/s

In [ ]:
om, X_fine = fourier(u, dt)                 # all masts
_, X_rep = fourier(report_hub, dt)
# your code here: the two spectra (course convention; the fine one averaged over all masts).
# Then read om_silent from the right-hand plot: the first ω above about 0.3 rad/s where report/fine drops and stays below 0.1
S_fine = None
S_rep = None
om_silent = None

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].loglog(*show(om, S_fine), lw=1, label="fine box")
ax[0].loglog(*show(om, np.maximum(S_rep, 1e-12)), lw=1, label="report's box")
ax[0].set_ylim(1e-7, None); ax[0].set_ylabel("S_u(ω) [m²/s]"); ax[0].legend()
ax[1].loglog(*show(om, S_rep / S_fine), lw=1, color="C2"); ax[1].axhline(0.1, ls="--", color="k", lw=1)
ax[1].set_ylabel("report / fine"); ax[1].set_ylim(1e-4, 10)
for a_ in ax:
    a_.axvline(om_silent, ls=":", color="k"); a_.set_xlabel("ω [rad/s]"); a_.grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()
print(f"variance: fine {np.var(u, axis=0).mean():.2f}, report {np.var(report_hub):.2f} m²/s²;  ω_silent ≈ {om_silent:.2f} rad/s")

## Task 4 — Coherence across the rotor: is Davenport right? (7')

The estimator is given (`coherence`). Davenport (eq. 7.1.11): $\mathrm{coh}(f, \Delta) = \exp(-a f \Delta / U)$, with $f$
in **Hz**. Write the model in terms of the $\omega$ you have, fit $a$ at 15 m, and use it to predict 30 and 45 m.

**Lab result 2:** a = … · At 45 m Davenport is (higher / lower) than the box → for the designer that is (safe / unsafe).

In [ ]:
om, X = fourier(u, dt)
coh = {j * dy: coherence(X, j, T) for j in (5, 10, 15)}

def davenport(om, a, sep):
    # your code here
    return None

sel = (om > 0.05) & (om < 3)
a_fit = curve_fit(lambda w, a: davenport(w, a, 15.0), om[sel], coh[15.0][sel], p0=[8])[0][0]
print(f"a (fitted at 15 m) = {a_fit:.1f}")
plt.figure(figsize=(7, 3.8))
for sep, c in coh.items():
    l, = plt.semilogx(*show(om, c), ".", ms=3, label=f"box, Δy = {sep:.0f} m")
    plt.semilogx(om[1:], davenport(om[1:], a_fit, sep), "-", color=l.get_color())
plt.xlim(0.02, 10); plt.ylim(0, 1.05); plt.xlabel("ω [rad/s]"); plt.ylabel("coherence"); plt.legend(); plt.grid(alpha=.3); plt.show()

---
### Into the game

Type **lab result 1** (ω_silent) and **lab result 2** (a) into the game when it asks for them. The slides then take both
numbers further: the coherence into the rotor's admittance, and ω_silent into the diagnosis.